# Запасной кандидат — `submission_04.csv` (V5 + median imputation)

Та же ансамблевая модель **V5** (SVC+HGB+ExtraTrees+2×MLP), но с **медианной импутацией** `eog_burst_index` (без модельного восстановления). CV macro-F1 = **0.8365**, public = **0.84266**.

Зачем хранить: это кандидат **другой природы** (без IterativeImputer). Если private неожиданно поднимет его выше основного — ноутбук воспроизводит его точно. Запускается и даёт тот же `submission_04.csv`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
sns.set_theme(style="whitegrid"); RANDOM_STATE=42
names={0:"Wake",1:"Light",2:"Deep",3:"REM"}
print("Среда готова. Только scikit-learn (без LightGBM/CatBoost).")

Среда готова. Только scikit-learn (без LightGBM/CatBoost).


In [2]:
EEG=["eeg_delta_power","eeg_theta_power","eeg_alpha_power","eeg_sigma_power","eeg_beta_power","eeg_gamma_power"]
def add_features(df):
    X=df.copy(); tot=X[EEG].clip(lower=0).sum(axis=1)+1e-6
    for b in EEG: X["rel_"+b]=X[b]/tot
    X["delta_beta"]=X["eeg_delta_power"]/(X["eeg_beta_power"].abs()+1e-6)
    X["theta_alpha"]=X["eeg_theta_power"]/(X["eeg_alpha_power"].abs()+1e-6)
    X["slow_dom"]=X["eeg_slow_osc_power"]+X["eeg_delta_power"]
    X["eog_burst_missing"]=df["eog_burst_index"].isna().astype(int)
    return X

In [3]:
train=pd.read_csv("train.csv"); test=pd.read_csv("test.csv"); sample=pd.read_csv("sample_submission.csv")
feat=[c for c in train.columns if c not in ("id","sleep_stage")]
X=add_features(train[feat]); y=train["sleep_stage"].values; X_test=add_features(test[feat])
print("train",train.shape,"test",test.shape,"| признаков после FE:",X.shape[1])

train (9000, 23) test (5000, 22) | признаков после FE: 31


In [4]:
def sc(m): return Pipeline([("s",StandardScaler()),("m",m)])
def V5(seed, imputer):
    return Pipeline([("imp",imputer()),("v",VotingClassifier([
        ("svc",sc(SVC(C=80,gamma=0.008,probability=True,random_state=seed))),
        ("hgb",HistGradientBoostingClassifier(random_state=seed,learning_rate=0.079,max_iter=240,
                 max_leaf_nodes=43,min_samples_leaf=24,l2_regularization=7.26)),
        ("et",ExtraTreesClassifier(n_estimators=430,max_features=0.89,min_samples_leaf=1,random_state=seed,n_jobs=-1)),
        ("mlp1",sc(MLPClassifier(hidden_layer_sizes=(128,64),alpha=1e-3,max_iter=400,early_stopping=True,random_state=seed))),
        ("mlp2",sc(MLPClassifier(hidden_layer_sizes=(200,100),activation="tanh",alpha=1e-3,max_iter=400,early_stopping=True,random_state=seed)))],
        voting="soft",n_jobs=-1))])

In [5]:
def median_imp(): return SimpleImputer(strategy="median")
cv=StratifiedKFold(5,shuffle=True,random_state=42)
print("V5 + median CV macro-F1 (seed42) = %.4f"%cross_val_score(V5(42,median_imp),X,y,cv=cv,scoring="f1_macro",n_jobs=-1).mean())

V5 + median CV macro-F1 (seed42) = 0.8367


**🔎** ~0.836 — ниже основного (0.844), т.к. median-импутация теряет восстановимый сигнал eog. Держим как hedge.

In [6]:
probs=np.zeros((len(X_test),4))
for s in [0,1,42]:
    m=V5(s,median_imp); m.fit(X,y); probs+=m.predict_proba(X_test)
pred=(probs/3).argmax(1)
pd.DataFrame({"id":test.id,"sleep_stage":pred}).to_csv("submission_04.csv",index=False)
print("submission_04.csv записан; распределение",[f'{names[i]}={(pred==i).mean()*100:.1f}%' for i in range(4)])

submission_04.csv записан; распределение ['Wake=22.4%', 'Light=25.7%', 'Deep=25.6%', 'REM=26.3%']


**🔎** Воспроизводит `submission_04.csv` (seed-bag 3 сида, median). Формат `id,sleep_stage`, 5000 строк.